# ToonForge — Free-GPU render (animated + sung + lip-sync)
Run on a **free GPU** (Kaggle 30h/wk · Colab · Lightning 22h/mo). Set the accelerator to **GPU** first.
Rotate: when one service's free quota runs out, run the same cells on another. See `docs/FREE_GPU.md`.


## 1. Code + GPU deps


In [ ]:
# Repo is public -> clones directly. (Private? run a cell first: import os; os.environ['GITHUB_TOKEN']='ghp_...'
# and change the URL below to https://${GITHUB_TOKEN}@github.com/...)
!rm -rf Video_Generator
!git clone https://github.com/4-7-9-0-6/Video_Generator.git
%cd Video_Generator/backend
!pip install -q -r requirements.txt
!pip install -q 'diffusers>=0.32' 'transformers>=4.44' accelerate sentencepiece 'imageio[ffmpeg]'
# ACE-Step (real singing) installs from GitHub, NOT pip — there is no 'acestep' package on PyPI:
!pip install -q 'git+https://github.com/ace-step/ACE-Step.git'

## 2. GPU check + your free keys


In [ ]:
import torch, os
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - set Accelerator=GPU')
os.environ['OPENROUTER_API_KEY']    = 'sk-or-...'      # free: openrouter.ai
os.environ['CLOUDFLARE_ACCOUNT_ID'] = '...'            # free: dash.cloudflare.com -> Workers AI
os.environ['CLOUDFLARE_API_TOKEN']  = '...'
os.environ['PROVIDER_IMAGE'] = 'cloudflare'   # free Flux stills
os.environ['PROVIDER_VIDEO'] = 'ltx_local'    # GPU animation
os.environ['PROVIDER_SVS']   = 'acestep_local' # GPU singing


## 3. Render (LTX animation + ACE-Step singing)
First run downloads the ACE-Step + LTX checkpoints once (a few GB).


In [ ]:
!python scripts/gpu_render.py --prompt 'a brave little robot lights up a neon city at night' \
    --style anime_cyberpunk --scenes 6 --out /kaggle/working/song.mp4


Download `song.mp4` from the output panel. ▶


## 4. (Optional) Lip-sync mode — SadTalker instead of scene animation


In [ ]:
!git clone https://github.com/OpenTalker/SadTalker && (cd SadTalker && bash scripts/download_models.sh)
!pip install -q face_alignment gfpgan kornia yacs pydub safetensors librosa


In [ ]:
import os
os.environ['SADTALKER_DIR']    = os.getcwd()+'/SadTalker'
os.environ['PROVIDER_LIPSYNC'] = 'sadtalker_local'
!python scripts/gpu_render.py --prompt 'a cheerful toddler sings a bath-time rhyme' \
    --style 3d_toddler_original --lipsync --out /kaggle/working/song_lipsync.mp4
